# 🖼️ Módulo 05 — Multimodal (Análisis de Imágenes)

> **Objetivo:** Aprender a enviar imágenes al agente combinando texto + imagen en un único mensaje.

## ¿Qué modelos soportan visión?

- `gpt-4o`, `gpt-4o-mini`, `gpt-4-turbo` (Azure OpenAI)
- Otros modelos multimodales compatibles

## API

```python
from agent_framework import Content, Message

message = Message(
    role="user",
    contents=[
        Content.from_text(text="Describe this image:"),
        Content.from_uri(uri="https://...", media_type="image/png"),
    ],
)
response = await agent.run([message])
```

Soporta tanto **URLs públicas** como **data-URIs base64** (para imágenes locales).


## ⚙️ Setup inicial

Esta celda carga la configuración de Azure OpenAI desde `appsettings.Development.json`
(o variables de entorno) y crea el cliente que reutilizaremos durante todo el notebook.

> 💡 Solo necesitas ejecutar esta celda **una vez** al abrir el notebook.


In [ ]:
# Aseguramos que la raíz del proyecto esté en sys.path para importar `helpers`
import sys
from pathlib import Path

_ROOT = Path.cwd()
# Si abriste el notebook desde la carpeta notebooks/, sube un nivel
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from agent_framework import Agent
from helpers.config import create_chat_client
print("✅ Helpers cargados. Endpoint:", create_chat_client().__class__.__name__)


## 1️⃣ Imagen por URL pública

Más simple: la URL viaja directamente al modelo. Útil cuando la imagen ya está alojada en internet.


In [ ]:
from agent_framework import Content, Message

agent = Agent(
    client=create_chat_client(),
    instructions=(
        "Eres un asistente de análisis de imágenes. Describe las imágenes de forma "
        "clara y concisa."
    ),
)

message = Message(
    role="user",
    contents=[
        Content.from_text(text="Describe esta imagen en una oración:"),
        Content.from_uri(
            uri="https://gelsoftcom.azurewebsites.net/images/Soldados001.png",
            media_type="image/png",
        ),
    ],
)

response = await agent.run([message])
print("✅ Análisis de imagen por URL:")
print(f"   {response.text}")


## 2️⃣ Imagen inline (base64)

Cuando la imagen está en disco (o se acaba de generar) y NO tiene URL pública, codifícala como data-URI.

```python
import base64
with open("foto.png", "rb") as f:
    b64 = base64.b64encode(f.read()).decode()
data_uri = f"data:image/png;base64,{b64}"
```

A continuación usamos un PNG mínimo de 1×1 píxel para demostrar el patrón sin necesitar archivos externos.


In [ ]:
# Pixel amarillo 1x1 en PNG (sólo para demo, en producción sería tu archivo real)
tiny_png_b64 = (
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlE"
    "QVR42mP8/5+hHgAHggJ/PchI7wAAAABJRU5ErkJggg=="
)
image_uri = f"data:image/png;base64,{tiny_png_b64}"

agent = Agent(client=create_chat_client(), instructions="Describe lo que ves en una oración corta. Responde en español.")

message = Message(
    role="user",
    contents=[
        Content.from_text(text="¿Qué hay en esta imagen?"),
        Content.from_uri(uri=image_uri, media_type="image/png"),
    ],
)

response = await agent.run([message])
print("✅ Análisis de imagen inline (base64):")
print(f"   {response.text}")


## 🎯 Resumen

| Origen | Sintaxis |
|--------|----------|
| URL pública | `Content.from_uri(uri="https://...", media_type="image/png")` |
| Imagen local | `Content.from_uri(uri="data:image/png;base64,...", media_type="image/png")` |

> 💡 **Tip:** combina varias imágenes en `contents=[...]` para que el agente las analice juntas
> (comparaciones antes/después, secuencias, etc.).

**Siguiente módulo →** [Módulo 06 — Conversaciones y Sesiones](./06_conversations_sessions.ipynb)
